# training_objective_handoff_v1

Notebook-first model-selection diagnostic. This combines the prior walk-forward weighted selector, repeated poor-window clusters, the latest path-level router result, and a fresh poor-window diagnostic for `recovery_reentry_conditional_vol_target_v1`.

Targets: relative excess >= 30pp, MaxDD <= 15%, no promotion without walk-forward evidence.

## External Basis

- [Qlib GitHub](https://github.com/microsoft/qlib): reproducible quant workflow across data, model, backtest, and deployment artifacts.
- [Qlib arXiv](https://arxiv.org/abs/2009.11189): AI-oriented quantitative investment workflow.
- [Probability of Backtest Overfitting](https://doi.org/10.21314/JCF.2016.322): repeated backtest selection requires OOS/walk-forward discipline.
- [Deflated Sharpe Ratio](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551): performance evidence should account for selection bias and multiple trials.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display
from src.soxl_quant.app.training_objective_handoff import (
    derive_training_objective_handoff,
    render_training_objective_handoff,
)

card = Path('runtime/model_selection/card/training_objective_handoff_v1')
paths = {
    'walkforward_scores': Path('runtime/model_selection/card/walkforward_weighted_objective_v1/walkforward_weighted_scores.csv'),
    'poor_window_clusters': Path('runtime/model_selection/card/oos_poor_window_factor_diff_aggregate_v1/aggregate/poor_window_clusters.csv'),
    'feature_groups': Path('runtime/execution/card/poor_window_mechanism_router_v1/feature_group_consensus.csv'),
    'router_selection': Path('runtime/execution/card/current_coverage_rebound_downtrend_router_v1/selection.json'),
    'candidate_poor_windows': card / 'diagnostics/recovery_reentry_conditional_vol_target_v1/poor_windows.csv',
    'candidate_feature_differences': card / 'diagnostics/recovery_reentry_conditional_vol_target_v1/feature_differences.csv',
}
pd.DataFrame([{'name': key, 'path': str(value), 'exists': value.exists()} for key, value in paths.items()])

,name,path,exists
0,walkforward_scores,runtime/model_selection/card/walkforward_weigh...,True
1,poor_window_clusters,runtime/model_selection/card/oos_poor_window_f...,True
2,feature_groups,runtime/execution/card/poor_window_mechanism_r...,True
3,router_selection,runtime/execution/card/current_coverage_reboun...,True
4,candidate_poor_windows,runtime/model_selection/card/training_objectiv...,True
5,candidate_feature_differences,runtime/model_selection/card/training_objectiv...,True


In [2]:
wf = pd.read_csv(paths['walkforward_scores'])
clusters = pd.read_csv(paths['poor_window_clusters'])
feature_groups = pd.read_csv(paths['feature_groups'])
router = json.loads(paths['router_selection'].read_text(encoding='utf-8'))
candidate_windows = pd.read_csv(paths['candidate_poor_windows'])
candidate_features = pd.read_csv(paths['candidate_feature_differences'])

handoff = derive_training_objective_handoff(
    walkforward_scores=wf,
    poor_window_clusters=clusters,
    feature_groups=feature_groups,
    router_selection=router,
)
display(Markdown(render_training_objective_handoff(handoff)))

# Training Objective Handoff

## Decision

- decision: `train_state_conditioned_multi_objective`
- promotion: `blocked`
- selected_diagnostic_candidate: `recovery_reentry_conditional_vol_target_v1`

## Objective Terms

- `walkforward_relative_return_gap_penalty`
- `walkforward_drawdown_gap_penalty`
- `high_vol_rebound_participation_floor`
- `downtrend_drawdown_state_cap`

## Risk Gap

- max_dd_gap=11.14pp, relative_gap=0.0pp, worst_window=`wf_2020`

## Primary Underparticipation Cluster

- cluster=`C003`, start=2026-02-04, end=2026-05-21, mean_excess=-167.30pp

## Stop Mechanisms

- `path_level_threshold_router_search`


In [3]:
display(wf.head(8))
display(clusters.head(5))
display(feature_groups.head(6))

,candidate,window_count,gate_pass_count,full_gate,weighted_objective_loss,mean_relative_return,min_relative_return,max_max_dd,mean_avg_pos,dominant_failure_type,worst_window,next_mechanism
0,recovery_reentry_conditional_vol_target_v1,3,0,False,237.495543,63.221576,55.796392,26.139320,0.282693,drawdown_control,wf_2020,drawdown_state_guard_or_sequence_risk_feature
1,recovery_reentry_cppi_floor_v1,3,0,False,276.141700,62.571551,55.796392,30.183184,0.303902,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature
2,recovery_reentry_regime_drawdown_guard_v1,3,0,False,378.452732,61.269911,55.796392,30.183184,0.314326,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature
3,recovery_reentry_2021_trendbreak_cap_v1,3,0,False,384.594038,44.753732,25.557493,28.204933,0.279729,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature
4,soxl_soxx_efficiency_chop_guard_v1,3,0,False,385.932397,42.024165,25.816839,30.512835,0.285571,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature
5,recovery_reentry_floor_v1,3,0,False,401.915992,57.416421,49.902600,30.183184,0.327356,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature
6,recovery_reentry_anchor_confirm_v1,3,0,False,408.862913,60.084194,55.796392,30.183184,0.314751,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature
7,v19reg_v22_budget_usage_bull_guard_v1,3,0,False,408.862913,54.877823,40.177280,30.183184,0.306417,drawdown_control,wf_2023_2024,drawdown_state_guard_or_sequence_risk_feature


,cluster_id,cluster_start,cluster_end,profile_count,window_count,profiles,mean_excess_return,worst_excess_return,mean_max_dd,max_max_dd,mean_avg_pos,mean_bad_score,max_bad_score,dominant_failure_type,primary_optimization_path
0,C002,2025-04-04,2025-07-22,5,30,"balanced_rule_residual_v19_reg,balanced_rule_r...",-160.502243,-199.393437,5.924443,16.324924,0.519568,166.426685,203.420490,underparticipation,reentry_participation_or_behavior_clone
1,C003,2026-02-04,2026-05-21,4,18,"balanced_rule_residual_v22,current_runtime_pap...",-167.300896,-198.889223,10.842927,24.118142,0.089418,178.143823,205.310696,underparticipation,reentry_participation_or_behavior_clone
2,C001,2020-03-18,2020-06-19,4,12,"balanced_rule_residual_v19_reg,balanced_rule_r...",-158.096497,-174.787005,8.683677,13.885509,0.514949,166.780175,182.311302,underparticipation,reentry_participation_or_behavior_clone


,feature_group,top_mentions,source_count,mean_abs_delta_z,features
0,volatility_path,30,7,1.266215,"jump_variation_5,realized_kurtosis_21,realized..."
1,trend_rebound,12,7,0.968214,"decay_63,mom_21,relative_strength_21"
2,downside_shape,6,6,0.784950,downside_ratio_21
3,other,6,6,0.784266,expectancy_proxy_21
4,tracking_error,1,1,1.445182,tracking_error_21
5,vix_regime,1,1,0.743305,vix_percentile_252


In [4]:
display(candidate_windows.head(12))
display(candidate_features.head(12))

,start,end,rows,strategy_return,buyhold_return,excess_return,max_dd,avg_pos,bad_score
0,2026-02-10,2026-05-11,63,25.600725,200.631509,-175.030784,6.421473,0.212055,181.452256
1,2026-02-05,2026-05-06,63,37.862930,211.455411,-173.592480,6.421473,0.234909,180.013953
2,2026-02-12,2026-05-13,63,24.653313,191.980998,-167.327685,6.421473,0.196819,173.749158
3,2026-02-13,2026-05-14,63,22.710293,188.264457,-165.554164,6.421473,0.189202,171.975637
4,2025-04-08,2025-07-09,63,64.546675,223.030298,-158.483624,11.499213,0.468268,169.982837
5,2025-04-21,2025-07-21,63,61.313720,215.729046,-154.415326,11.499213,0.530900,165.914538
6,2026-02-09,2026-05-08,63,21.244513,176.339225,-155.094712,6.421473,0.219673,161.516185
7,2026-02-23,2026-05-21,63,18.731376,170.862432,-152.131056,7.259839,0.156349,159.390894
8,2025-04-17,2025-07-18,63,58.402182,197.065223,-138.663040,11.499213,0.524979,150.162253
9,2026-02-20,2026-05-20,63,16.404784,158.083736,-141.678953,7.259839,0.158730,148.938791


,feature,bad_mean,reference_mean,delta,delta_z,bad_coverage,reference_coverage,spearman_to_excess,abs_delta_z
0,vol_of_vol_21,0.226385,0.116345,0.110039,1.194260,1.0,1.0,-0.023747,1.194260
1,jump_variation_5,1.149860,0.119732,1.030128,1.193689,1.0,1.0,0.005793,1.193689
2,realized_vol_63,1.371953,1.015122,0.356831,1.093212,1.0,1.0,-0.009067,1.093212
3,realized_kurtosis_21,1.522293,0.269898,1.252395,0.936568,1.0,1.0,-0.032719,0.936568
4,mom_21,0.273394,0.060919,0.212475,0.813171,1.0,1.0,-0.186142,0.813171
5,relative_strength_21,1.787618,0.380245,1.407374,0.802635,1.0,1.0,-0.184676,0.802635
6,downside_ratio_21,0.402805,0.470870,-0.068066,-0.795139,1.0,1.0,0.189254,0.795139
7,expectancy_proxy_21,0.215911,0.067499,0.148412,0.794799,1.0,1.0,-0.209154,0.794799
8,trend_slope_21,0.102205,0.013746,0.088459,0.770250,1.0,1.0,-0.005155,0.770250
9,vix_percentile_252,0.685937,0.453480,0.232457,0.722150,1.0,1.0,0.113525,0.722150


In [5]:
card.mkdir(parents=True, exist_ok=True)
(card / 'training_objective_handoff.json').write_text(json.dumps(handoff, ensure_ascii=True, indent=2), encoding='utf-8')
(card / 'training_objective_handoff.md').write_text(render_training_objective_handoff(handoff), encoding='utf-8')
pd.DataFrame([{'name': key, 'path': str(value), 'exists': value.exists()} for key, value in paths.items()]).to_csv(card / 'source_manifest.csv', index=False)
print('wrote', card / 'training_objective_handoff.json')

wrote runtime/model_selection/card/training_objective_handoff_v1/training_objective_handoff.json


## Runtime Conclusion

Decision: `train_state_conditioned_multi_objective`. Promotion remains `blocked`.

Evidence chain:

- Walk-forward weighted screen selected diagnostic candidate `recovery_reentry_conditional_vol_target_v1`, but it is not promotable: worst window `wf_2020` has MaxDD `26.14%`, which is `11.14pp` above the 15% target.
- Shared poor-window aggregation still points to high-vol rebound underparticipation: `C003` from `2026-02-04` to `2026-05-21` has mean excess `-167.30pp` and mean position `0.089`.
- New selected-candidate poor-window diagnostic confirms the same mechanism: the top poor windows are 2026/2025 rebound windows with excess below -130pp while MaxDD stays below 12%, so the low-loss candidate is still missing upside participation where buy-and-hold accelerates.
- The current coverage rebound/downtrend router improved the 2026 rebound cluster by `51.36pp` but still failed strict plus walk-forward gates, so another path-level threshold sweep is now a duplicated mechanism.

Next valid loop:

Train or select with an explicit state-conditioned multi-objective: `walkforward_relative_return_gap_penalty`, `walkforward_drawdown_gap_penalty`, `high_vol_rebound_participation_floor`, and `downtrend_drawdown_state_cap`. Keep this as training-time / checkpoint-selection evidence, not runtime promotion.
